# 실습 2. 지도학습 분류 모델 (Day 1 / 150분)

> 시나리오: **리뷰 → 감성(긍정/부정) 분류** — 입력은 실습 1의 `reviews_clean.parquet`
>
> 이 sklearn 베이스라인은 Day 2 실습 3에서 **LLM 분류와 비교**하는 기준선이 된다.

## Task
1. 라벨 생성 + train/test 분할 (stratify)
2. **Baseline**: TF-IDF + LogisticRegression
3. **개선 1**: ngram (1,2) + min_df / max_df 튜닝
4. **개선 2**: RandomForest / GradientBoosting 비교
5. **혼동 행렬** — 어디서 틀리는가
6. 틀린 샘플 10개를 직접 읽고 **오분류 원인 분류**

In [1]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
# 4~5 -> 긍정(1), 1~2 -> 부정(0), 3 -> 제외
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]
df["label"] = (df["rating"] >= 4).astype(int)
print(df["label"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/reviews_clean.parquet'

## 1) 분할 (stratify 로 클래스 비율 유지)

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"].fillna("")
y = df["label"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

NameError: name 'df' is not defined

## 2) Baseline — TF-IDF + LogisticRegression (Pipeline 으로 누수 방지)

> **TF-IDF** = 텍스트를 "어떤 단어가 이 문서에서 얼마나 특징적인가"의 **숫자 벡터**로 바꾸는 것.
> 모델은 글자가 아니라 이 숫자를 학습한다. `Pipeline` 으로 벡터화→학습을 묶어 test 누수를 막는다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

pipe = Pipeline([
    ("vec", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
print(classification_report(y_te, pipe.predict(X_te), digits=3))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000       671
           1      1.000     1.000     1.000       950

    accuracy                          1.000      1621
   macro avg      1.000     1.000     1.000      1621
weighted avg      1.000     1.000     1.000      1621



## 3~4) 개선 — 모델 비교표

baseline 의 `vec` / `clf` 만 바꿔 끼워 **여러 후보를 같은 루프로 비교**한다.
아래 셀은 그대로 실행되며, 정확도·F1·학습시간 표를 만든다.
- **개선 1**: TF-IDF 에 `ngram_range=(1,2)` + `min_df` / `max_df`
- **개선 2**: `RandomForestClassifier`, `GradientBoostingClassifier`

> 비교는 **속도를 위해 학습셋 일부(3,000건)** 로 돌린다. 순위 경향은 전체와 같고,
> 전체로 돌리면 더 정확하지만 GradientBoosting 은 수 분 걸릴 수 있다.
> 표를 만든 뒤 **왜 그런 순위인지** 한 줄로 해석하는 것이 이 Task 의 핵심이다.

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

# 비교는 속도를 위해 학습셋 일부(3,000건)만 사용 — 순위 경향은 전체와 동일
N_FIT = 3000
X_fit, y_fit = X_tr.iloc[:N_FIT], y_tr.iloc[:N_FIT]

# 비교할 후보: 이름 → 파이프라인 (vec/clf 만 바꿔 끼운다)
candidates = {
    "LogReg (baseline)": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "LogReg + ngram(1,2)": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.9)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "RandomForest": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
    "GradientBoosting": Pipeline([
        ("vec", TfidfVectorizer(max_features=5000)),
        ("clf", GradientBoostingClassifier(random_state=42)),
    ]),
}

rows = []
for name, model in candidates.items():
    print(f"· {name} 학습 중...")                # 멈춘 게 아니라 학습 중
    t0 = time.perf_counter()
    model.fit(X_fit, y_fit)                       # 서브셋으로 빠르게 비교
    train_sec = time.perf_counter() - t0
    pred = model.predict(X_te)                    # 평가는 전체 테스트셋
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "train_sec": train_sec,
    })

report = pd.DataFrame(rows).set_index("model").round(3)
report.sort_values("f1", ascending=False)
# TODO: 1위/꼴찌 모델이 왜 그런지 한 줄 해석 (선형 vs 트리, ngram 효과, 학습시간 trade-off)

## 5) 혼동 행렬

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = pipe.predict(X_te)
print(confusion_matrix(y_te, y_pred))

[[671   0]
 [  0 950]]


## 6) 오분류 케이스 직접 읽기 (반어 / 짧은 텍스트 / 도메인어 ...)

In [ ]:
wrong = X_te[y_te != y_pred]
for t in wrong.head(10):
    print("-", t)
# TODO: 오분류 원인 3가지 카테고리로 메모